In [1]:
import math
import torch

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
    set_seed,
)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# We use the Wikitext-2 raw dataset from Hugging Face
raw_ds = load_dataset("wikitext", "wikitext-2-raw-v1")

print(raw_ds)

train_raw = raw_ds["train"]

def clean_example(example):
    text = example["text"].strip()
    return {"text": text}

# Basic cleaning: strip whitespace
train_clean = train_raw.map(clean_example)

# Filter out empty lines and some obvious headers
def is_valid(example):
    txt = example["text"]
    if len(txt) == 0:
        return False
    if txt.startswith(" =") or txt.startswith("= "):
        return False
    return True

dataset = train_clean.filter(is_valid)

# subsample for speed
max_samples = 5000
if len(dataset) > max_samples:
    dataset = dataset.select(range(max_samples))

print(f"Number of cleaned training lines: {len(dataset)}")
print("Sample example:")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Number of cleaned training lines: 5000
Sample example:
{'text': 'Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .'}


In [3]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# GPT-2 doesn't have a pad token by default; we reuse EOS
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}, id: {tokenizer.pad_token_id}")

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding=False,        # we'll handle padding later in the collate_fn
        truncation=False,     # we want to keep full text, grouping will handle length
        return_attention_mask=True,
    )

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

# Filter out any weird empty cases after tokenization
tokenized = tokenized.filter(lambda ex: len(ex["input_ids"]) > 0)

print(tokenized)
print("Example tokenized entry:")
print({k: v[:20] for k, v in tokenized[0].items()})

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer vocab size: 50257
Pad token: <|endoftext|>, id: 50256


Tokenizing:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5000
})
Example tokenized entry:
{'input_ids': [10445, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791, 47398, 17740, 357, 4960, 1058, 10545, 230, 99, 161, 254], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [4]:
# For language model training, we usually create fixed-length sequences
block_size = 128

def group_texts(examples):
    # Concatenate all input_ids and attention_masks in the batch
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks = sum(examples["attention_mask"], [])

    # Drop remainder so length is divisible by block_size
    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks = concatenated_masks[:total_len]

    # Chunk into blocks
    result_input_ids = [
        concatenated_inputs[i : i + block_size]
        for i in range(0, total_len, block_size)
    ]
    result_masks = [
        concatenated_masks[i : i + block_size]
        for i in range(0, total_len, block_size)
    ]

    return {
        "input_ids": result_input_ids,
        "attention_mask": result_masks,
    }

lm_ds = tokenized.map(
    group_texts,
    batched=True,
    desc=f"Grouping tokens into blocks of {block_size}",
)

print(lm_ds)
print("Example grouped entry:")
print({k: v[:10] for k, v in lm_ds[0].items()})

Grouping tokens into blocks of 128:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 4976
})
Example grouped entry:
{'input_ids': [10445, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [5]:
# Small validation set for sanity checking
split = lm_ds.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(f"Train sequences: {len(train_ds)}")
print(f"Validation sequences: {len(val_ds)}")

Train sequences: 4727
Validation sequences: 249


In [6]:
def collate_fn(batch):
    """
    Collate function for causal language modeling:
    - Stacks input_ids and attention_mask into tensors.
    - Uses input_ids as labels (model will predict next token).
    """
    input_ids = torch.tensor(
        [ex["input_ids"] for ex in batch],
        dtype=torch.long,
    )
    attention_mask = torch.tensor(
        [ex["attention_mask"] for ex in batch],
        dtype=torch.long,
    )

    # For causal LM, labels are usually the same as input_ids
    labels = input_ids.clone()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

batch_size = 8

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

# Inspect a single batch
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([8, 128])
attention_mask torch.Size([8, 128])
labels torch.Size([8, 128])


In [8]:
model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))  # pad token added
model.to(device)

learning_rate = 5e-5
num_epochs = 1

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

num_update_steps_per_epoch = len(train_loader)
num_training_steps = num_epochs * num_update_steps_per_epoch

lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

global_step = 0
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        global_step += 1

        if step % 50 == 0:
            print(f"Epoch {epoch} | step {step}/{len(train_loader)} | loss = {loss.item():.4f}")

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Mean training loss: {avg_loss:.4f}")

Epoch 0 | step 0/591 | loss = 4.4175
Epoch 0 | step 50/591 | loss = 4.1004
Epoch 0 | step 100/591 | loss = 3.9528
Epoch 0 | step 150/591 | loss = 3.6444
Epoch 0 | step 200/591 | loss = 4.0282
Epoch 0 | step 250/591 | loss = 3.5995
Epoch 0 | step 300/591 | loss = 3.5620
Epoch 0 | step 350/591 | loss = 3.8677
Epoch 0 | step 400/591 | loss = 3.7581
Epoch 0 | step 450/591 | loss = 3.7808
Epoch 0 | step 500/591 | loss = 3.6609
Epoch 0 | step 550/591 | loss = 3.7074
Epoch 0 finished. Mean training loss: 3.6993


In [9]:
model.eval()
val_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        val_loss += loss.item()
        num_batches += 1

val_loss = val_loss / max(1, num_batches)
perplexity = math.exp(val_loss)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {perplexity:.2f}")

Validation loss: 3.4706
Validation perplexity: 32.16


In [10]:
prompt = "In a distant future, artificial intelligence"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_length=60,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
    )

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a distant future, artificial intelligence was thought to be the future of human communication . It had begun to develop its own language , but was never fully developed . In 1943 , a Canadian scientist named Harold R. B. Sabin published a paper describing artificial intelligence as " an extension of our species "
